# 02 - Point-in-time features + rules baseline

Manchester.Ai_IBM Z Datathon 2026 Project. Runs on IBM LinuxONE.

**The question we answer:** the moment a payment lands in an account, does it look like that account is being used as a money mule?

**The rule that keeps us honest:** every feature for a payment uses only transactions from *strictly before* that payment's timestamp. Nothing from the future leaks in, so the model only knows what a bank would know at that moment.

In [1]:
import os, json, time, gc
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

SHARED = os.path.expanduser('~/shared')
t0 = time.time()
df = pd.read_pickle(f'{SHARED}/hi_small_slim_usd.pkl')
print(df.shape, f'loaded in {time.time() - t0:.1f}s')

(4486418, 16) loaded in 1.7s


## 2. The point-in-time engine
`window_stats` answers questions like *"how many payments did this account receive in the 24 hours before this payment?"* for all ~4.5M rows at once.

It sorts events by (account, time) and uses binary search, so each feature takes seconds rather than a slow per-account loop. Events at the same minute as the current payment are excluded, so there is no leakage.

In [2]:
# useful for feature generation

def window_stats(q_id, q_t, e_id, e_t, window=None, e_val=None):
    """For each query row: count (and optionally sum of e_val) of events for the same
    account with time in [q_t - window, q_t). window=None means all earlier history."""
    q_id = np.asarray(q_id, dtype='int64'); q_t = np.asarray(q_t, dtype='int64')
    e_id = np.asarray(e_id, dtype='int64'); e_t = np.asarray(e_t, dtype='int64')
    big = np.int64(max(q_t.max(), e_t.max()) + 1)
    ek = e_id * big + e_t
    order = np.argsort(ek, kind='stable')
    ek = ek[order]
    lo_t = np.zeros_like(q_t) if window is None else np.maximum(q_t - window, 0)
    r = np.searchsorted(ek, q_id * big + q_t, side='left')
    l = np.searchsorted(ek, q_id * big + lo_t, side='left')
    cnt = (r - l).astype('int32')
    if e_val is None:
        return cnt
    cs = np.concatenate([[0.0], np.cumsum(np.asarray(e_val, dtype='float64')[order])])
    return cnt, (cs[r] - cs[l]).astype('float32')

# Tiny sanity test on hand-made data: account 7 receives at t=0, 10, 10, 50
_c = window_stats(q_id=[7, 7, 7, 7], q_t=[0, 10, 10, 50], e_id=[7, 7, 7, 7], e_t=[0, 10, 10, 50], window=45)
assert list(_c) == [0, 1, 1, 2], _c   # same-minute events excluded; window [5, 50) at t=50 holds t=10, 10
print('window_stats sanity check passed')

window_stats sanity check passed


## 3. Build features
Three groups, matching the pitch: **the payment**, **the receiving account** (the possible mule) and **the sender** (the possible recruiter).

In [3]:
N_ACCT = len(df['sender'].cat.categories)
t0 = time.time()
H, D = 3600, 86400
s, r, t = df['s_id'].values, df['r_id'].values, df['t'].values
amt_paid = df['amount_paid_usd'].values
amt_recv = df['amount_received_usd'].values
f = pd.DataFrame(index=df.index)

# --- The payment itself ---
f['log_amount'] = np.log1p(amt_paid).astype('float32')
f['payment_format'] = df['payment_format'].cat.codes.astype('int8')
f['payment_currency'] = df['payment_currency'].cat.codes.astype('int8')
# compare currencies as small numbers, not text (CHANGED)
cur_cats = df['payment_currency'].cat.categories
f['cross_currency'] = (df['payment_currency'].cat.codes.values
                       != pd.Categorical(df['receiving_currency'], categories=cur_cats).codes).astype('int8')
f['cross_bank'] = (df['from_bank'] != df['to_bank']).astype('int8')
f['hour'] = df['timestamp'].dt.hour.astype('int8')

new_pair = (~df.duplicated(['s_id', 'r_id'])).values          # first time this sender has paid this receiver
f['new_pair'] = new_pair.astype('int8')
pair = pd.factorize(s * N_ACCT + r)[0]                          # CHANGED: N_ACCT instead of acct
f['pair_prior_cnt'] = window_stats(pair, t, pair, t)

# --- The receiving account (possible mule) ---
f['r_in_cnt_all'], r_in_sum_all = window_stats(r, t, r, t, None, amt_recv)
f['r_in_cnt_24h'], r_in_sum_24h = window_stats(r, t, r, t, D, amt_recv)
f['r_in_cnt_1h'] = window_stats(r, t, r, t, H)
f['r_out_cnt_all'] = window_stats(r, t, s, t)                              # has it ever sent money?
f['r_out_cnt_24h'], r_out_sum_24h = window_stats(r, t, s, t, D, amt_paid)  # money leaving recently
f['r_distinct_senders'] = window_stats(r, t, r[new_pair], t[new_pair])
f['r_new_senders_24h'] = window_stats(r, t, r[new_pair], t[new_pair], D)  # fan-in
avg_in = np.where(f['r_in_cnt_all'] > 0, r_in_sum_all / np.maximum(f['r_in_cnt_all'], 1), np.nan)
f['amt_vs_r_avg'] = np.where(np.isnan(avg_in), -1, amt_recv / (avg_in + 1)).astype('float32')
f['r_passthrough_24h'] = (r_out_sum_24h / (r_in_sum_24h + 1)).astype('float32')

# --- The sender (possible recruiter, or a mule forwarding money on) ---
f['s_out_cnt_all'] = window_stats(s, t, s, t)
f['s_out_cnt_24h'], s_out_sum_24h = window_stats(s, t, s, t, D, amt_paid)
f['s_in_cnt_24h'], s_in_sum_24h = window_stats(s, t, r, t, D, amt_recv)   # money passing through the sender too
f['s_distinct_receivers'] = window_stats(s, t, s[new_pair], t[new_pair])
f['s_new_receivers_24h'] = window_stats(s, t, s[new_pair], t[new_pair], D) # fan-out
# NEW: share of recently received money now going out (including this payment)
f['s_passthrough_24h'] = ((s_out_sum_24h + amt_paid) / (s_in_sum_24h + 1)).astype('float32')

# --- Account age: hours since first seen anywhere (CHANGED: lighter version) ---
first_seen = np.full(N_ACCT, np.iinfo('int64').max, dtype='int64')
np.minimum.at(first_seen, s, t)
np.minimum.at(first_seen, r, t)
f['r_age_h'] = ((t - first_seen[r]) / H).astype('float32')
f['s_age_h'] = ((t - first_seen[s]) / H).astype('float32')

FEATURES = list(f.columns)
print(f'{len(FEATURES)} features for {len(f):,} payments in {time.time() - t0:.1f}s')

# free the big temporary arrays
del (pair, new_pair, first_seen, avg_in, r_in_sum_all, r_in_sum_24h,
     r_out_sum_24h, s_out_sum_24h, s_in_sum_24h)
gc.collect()

25 features for 4,486,418 payments in 59.3s


0

In [4]:
f.sample(8, random_state=0).T

,3745378,4353619,4011002,2813267,1131601,1245342,45342,2225447
log_amount,6.096811,3.298306,9.093138,8.571064,7.051986,7.901714,10.836891,4.921776
payment_format,4.000000,4.000000,0.000000,0.000000,3.000000,3.000000,0.000000,4.000000
payment_currency,0.000000,9.000000,3.000000,8.000000,8.000000,8.000000,3.000000,5.000000
cross_currency,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
cross_bank,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
hour,3.000000,7.000000,13.000000,5.000000,16.000000,20.000000,0.000000,0.000000
new_pair,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
pair_prior_cnt,12.000000,18.000000,16.000000,14.000000,3.000000,0.000000,1.000000,0.000000
r_in_cnt_all,15.000000,19.000000,23.000000,28.000000,8.000000,4.000000,1.000000,0.000000
r_in_cnt_24h,2.000000,2.000000,6.000000,8.000000,3.000000,2.000000,1.000000,0.000000


## 4. Time-based split
Day 1 is warm-up only (every account looks brand new on day 1). We train on earlier days and test on later days, never a random split, so the test mimics the real future.

In [5]:
# train / val / test split based on day of transfer
# 1 Sep = warmup (no history yet), 2-6 train, 7-8 val, 9-10 test (+ live demo stream)
day = df['timestamp'].dt.day.values
data = f                      # same table, no second copy
del f
data['split'] = pd.Categorical(
    np.select([day == 1, day <= 6, day <= 8], ['warmup', 'train', 'val'], default='test'),
    categories=['warmup', 'train', 'val', 'test'])
data['y'] = df['is_laundering'].values.astype('int8')

# reference columns for the demo and explanations (NOT model features)
data['timestamp'] = df['timestamp'].values
data['sender'] = df['sender'].values                      # stays compact (category)
data['receiver'] = df['receiver'].values
data['amount_paid_usd'] = df['amount_paid_usd'].values
data['payment_format_name'] = df['payment_format'].values # category, no astype(str)

# make sure nothing that isn't a feature slipped into FEATURES
forbidden = {'y', 'split', 'timestamp', 'sender', 'receiver', 'amount_paid_usd',
             'payment_format_name', 'is_laundering', 's_id', 'r_id', 't'}
assert not forbidden & set(FEATURES), forbidden & set(FEATURES)

summary = data.groupby('split', observed=True)['y'].agg(rows='size', laundering='sum', rate='mean')
display(summary)
display(data[data['split'] == 'train'].head(5).T)   # sideways view of a few training rows

,rows,laundering,rate
split,,,
warmup,607511,321,0.000528
train,2089024,2203,0.001055
val,944726,1036,0.001097
test,845157,954,0.001129


,607511,607512,607513,607514,607515
log_amount,11.529788,1.391282,7.760987,9.848736,1.495149
payment_format,0,4,0,4,4
payment_currency,8,8,8,8,8
cross_currency,0,0,0,0,0
cross_bank,1,1,1,1,1
hour,0,0,0,0,0
new_pair,0,1,0,1,1
pair_prior_cnt,2,0,2,0,0
r_in_cnt_all,5,1,6,2,3
r_in_cnt_24h,5,1,6,2,3


In [6]:
os.makedirs(f'{SHARED}/models', exist_ok=True)
json.dump({'features': FEATURES}, open(f'{SHARED}/models/features.json', 'w'), indent=2)

model_cols = FEATURES + ['y']
ref_cols = ['timestamp', 'sender', 'receiver', 'amount_paid_usd', 'payment_format_name']

data.loc[data['split'] == 'train', model_cols].reset_index(drop=True).to_pickle(f'{SHARED}/feat_train.pkl')
data.loc[data['split'] == 'val', model_cols].reset_index(drop=True).to_pickle(f'{SHARED}/feat_val.pkl')
data.loc[data['split'] == 'test', model_cols + ref_cols].reset_index(drop=True).to_pickle(f'{SHARED}/feat_test.pkl')
print('saved feat_train.pkl, feat_val.pkl, feat_test.pkl and models/features.json')

saved feat_train.pkl, feat_val.pkl, feat_test.pkl and models/features.json


## 5. Rules baseline
What a simple hand-written bank rule would catch. The model must clearly beat this.

**Metric:** a bank can only review a small share of payments, so we flag the riskiest **1%** and ask: what share of all laundering payments are in that 1%? (recall at a 1% alert rate). We also report PR-AUC, which suits very rare positives.

In [7]:
# Baseline: a hand-written bank rulebook, evaluated on the test days (9-10 Sept).
# The model must beat these numbers with the same 1% of payments flagged.

def evaluate(y, score, rate=0.01):
    y = np.asarray(y); score = np.asarray(score, dtype='float64')
    k = max(1, int(len(y) * rate))
    top = np.argsort(-score, kind='stable')[:k]
    return {
        'recall@1%': y[top].sum() / max(y.sum(), 1),
        'precision@1%': y[top].mean(),
        'pr_auc': average_precision_score(y, score),
        'base_rate': y.mean(),
    }

fmt_cats = df['payment_format'].cat.categories
ach = int(np.where(fmt_cats == 'ACH')[0][0]) if 'ACH' in fmt_cats else -1

def rules_score(d):
    flags = (
        (d['payment_format'] == ach).astype(int)        # most laundering in this data moves by ACH
        + d['new_pair']                                  # first payment between these two accounts
        + (d['s_new_receivers_24h'] >= 3).astype(int)    # sender fanning out to new accounts
        + (d['r_new_senders_24h'] >= 2).astype(int)      # receiver collecting from new senders
        + (d['r_age_h'] < 24).astype(int)                # brand-new receiving account
    )
    return flags + 1e-3 * d['log_amount']                # tiny tie-breaker on amount

rule_cols = ['payment_format', 'new_pair', 's_new_receivers_24h', 'r_new_senders_24h',
             'r_age_h', 'log_amount', 'y']
test = data.loc[data['split'] == 'test', rule_cols]
baseline = evaluate(test['y'], rules_score(test))

# save for the model comparison slide
json.dump({k: float(v) for k, v in baseline.items()},
          open(f'{SHARED}/models/baseline_rules.json', 'w'), indent=2)

pd.Series(baseline).round(4)

recall@1%       0.2568
precision@1%    0.0290
pr_auc          0.0236
base_rate       0.0011
dtype: float64

## 6. Save for the modelling notebook